# Lesson 03 - Agentic Design Patterns

यस पाठमा, हामी प्रभावकारी AI एजेन्टहरू निर्माणका लागि तीन आधारभूत डिजाइन ढाँचाहरू अन्वेषण गर्नेछौं:

1. **स्पष्ट एजेन्ट निर्देशनहरू** — एजेन्टको व्यवहारलाई मार्गनिर्देशन गर्ने सटीक, भूमिका-परिभाषित प्रॉम्प्टहरू तयार पार्ने
2. **Pydantic मोडेलहरूसँग संरचित आउटपुट** — एजेन्टहरूले पूर्वानुमानित, प्रमाणित डेटा फिर्ता गर्ने सुनिश्चित गर्ने
3. **एकल जिम्मेवारी एजेन्टहरू** — फोकस गरिएको एजेन्टहरू डिजाइन गर्ने जो प्रत्येकले एक कुरा राम्रोसँग गर्छन्

हामी प्रत्येक ढाँचालाई एक **यात्रा गन्तव्य सिफारिस गर्ने** परिदृश्यमा लागू गर्नेछौं, क्रमशः एक यस्तो प्रणाली निर्माण गर्दै जसले गन्तव्यहरू सिफारिस गर्न, उपलब्धता जाँच गर्न, र व्यवस्थापन सम्हाल्न सक्छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## नमुना १: स्पष्ट एजेन्ट निर्देशनहरू

सबैभन्दा प्रभावकारी नमुना सबैभन्दा सरल पनि हो: तपाईंको एजेन्टका लागि स्पष्ट, विस्तृत निर्देशनहरू लेख्नु।

राम्रो निर्देशनहरूले निर्धारण गर्छन्:
- **को** एजेन्ट हो (व्यक्ति र स्वर)
- **के** यो गर्नुपर्छ (चरण-दर-चरण जिम्मेवारीहरू)
- **कसरी** यसको व्यवहार हुनुपर्छ (सीमाहरू र शैली)

तल, हामी एक यात्रा कन्सिएर्ज एजेन्ट बनाउँछौं जसले प्रत्येक प्रतिक्रिया बनाउँदा स्पष्ट निर्देशनहरू पालना गर्छ।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic मोडलसँग संरचित आउटपुट

फ्री-फर्म टेक्स्ट संवादका लागि उपयोगी छ, तर डाउनस्ट्रीम प्रणालीहरूलाई संरचित डाटा आवश्यक पर्छ।  
**Pydantic मोडलहरू** लाई **टुल फङ्सन** सँग जोडी हामीले गर्न सक्छौं:

- एजेन्टको आउटपुटको लागि बिल्कुलै निर्धारण गरिएको स्किमा परिभाषित गर्ने
- प्रतिक्रिया स्वतः प्रमाणीकरण गर्ने
- एजेन्टका परिणामलाई अनुप्रयोग तर्कमा भरपर्दो रूपमा समाहित गर्ने

हामीले एउटा यस्ता टुल पनि प्रस्तुत गर्छौं जसले गन्तव्य विवरणहरू फिर्ता गर्छ ताकि एजेन्टले आफ्ना सिफारिसहरू वास्तविक डाटामा आधारित राखोस्।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: एकल जिम्मेवारी एजेन्टहरू

जटिल कार्यहरूले धेरै केन्द्रित एजेन्टहरूमा काम विभाजन गर्दा फाइदा पुर्‍याउँछ, प्रत्येकसँग एकल जिम्मेवारी हुन्छ:

- एउटा **गन्तव्य विशेषज्ञ** जसले स्थानहरू र उपलब्धताबारे जान्दछ
- एउटा **लजिस्टिक्स योजनाकार** जसले उडान, होटेल, र यात्रा तालिकाहरू व्यवस्थापन गर्छ

यसले सफ्टवेयर इञ्जिनियरिङ सिद्धान्त *चासोहरूको पृथक्करण* लाई प्रतिबिम्बित गर्छ — प्रत्येक एजेन्टलाई स्वतन्त्र रूपमा परीक्षण, मर्मत, र सुधार गर्नु सजिलो हुन्छ।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

यस पाठमा हामीले यात्रा सिफारिसकर्ता परिदृश्यमा तीन वटा एजेन्टिक डिजाइन ढाँचाहरू लागू गर्यौं:

| ढाँचा | प्रमुख विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देशनहरू** | व्यक्तित्व, जिम्मेवारीहरू, र सीमाहरू पहिल्यै परिभाषित गर्नुहोस् | लगातार, ब्रान्डअनुकूल एजेन्ट व्यवहार |
| **संरचित आउटपुट** | जवाफ स्वरूपका लागि Pydantic मोडेलहरू प्रयोग गर्नुहोस् | मान्य, मेशिन-पठनीय परिणामहरू |
| **एकल जिम्मेवारी** | प्रत्येक एजेन्टलाई एक केन्द्रित काम दिनुहोस् | परीक्षण, मर्मत, र संयोजन गर्न सजिलो |

यी ढाँचाहरू स्वाभाविक रूपमा संयोजन हुन्छन् — तपाईं स्पष्ट निर्देशनहरूलाई संरचित आउटपुटसँग एकल-jimmedari agent भित्र संयुक्त गरेर बलियो, उत्पादन-तयार प्रणालीहरू निर्माण गर्न सक्नुहुन्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यो दस्तावेज़ [Co-op Translator](https://github.com/Azure/co-op-translator) भन्ने एआई अनुवाद सेवा प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयासरत भएता पनि, कृपया जानकार हुनुहोस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुनसक्छन्। मूल दस्तावेज आफ्नो मातृभाषामा आधिकारिक स्रोत मानिनेछ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा गलत व्याख्याप्रति हामी जिम्मेवार हुनेछैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
